# 00 — Data Inventory

Este notebook realiza o inventário inicial dos dados utilizados no projeto
Retail Demand Forecasting Pipeline.

Objetivos:
- Localizar os arquivos brutos;
- Identificar tabelas disponíveis;
- Analisar dimensões, colunas e tipos;
- Identificar possíveis chaves e granularidades;
- Documentar a estrutura dos dados antes da construção da camada Bronze.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "favorita"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data root: {RAW_ROOT}")

Project root: /home/vitoria/retail-demand-forecasting-pipeline
Raw data root: /home/vitoria/retail-demand-forecasting-pipeline/data/raw/favorita


In [2]:
# Definindo os caminhos principais do projeto
train_files = list(RAW_ROOT.rglob("train.csv"))

if not train_files:
    raise FileNotFoundError(
        "train.csv não encontrado. Faça o download dos dados antes de continuar."
    )

DATA_PATH = train_files[0].parent

print(f"Dataset path: {DATA_PATH}")

Dataset path: /home/vitoria/retail-demand-forecasting-pipeline/data/raw/favorita


In [3]:
# Localizando automaticamente a pasta que contém os arquivos do dataset
csv_files = sorted(DATA_PATH.glob("*.csv"))

for file in csv_files:
    print(file.name)

holidays_events.csv
oil.csv
sample_submission.csv
stores.csv
test.csv
train.csv
transactions.csv


In [4]:
# Carregando as principais tabelas utilizadas no projeto
train = pd.read_csv(DATA_PATH / "train.csv", parse_dates=["date"])
stores = pd.read_csv(DATA_PATH / "stores.csv")
transactions = pd.read_csv(DATA_PATH / "transactions.csv", parse_dates=["date"])
holidays = pd.read_csv(DATA_PATH / "holidays_events.csv", parse_dates=["date"])
oil = pd.read_csv(DATA_PATH / "oil.csv", parse_dates=["date"])

In [5]:
# Verificando as dimensões de cada tabela carregada
datasets = {
    "train": train,
    "stores": stores,
    "transactions": transactions,
    "holidays": holidays,
    "oil": oil,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

train: (3000888, 6)
stores: (54, 5)
transactions: (83488, 3)
holidays: (350, 6)
oil: (1218, 2)


In [6]:
# Verificando as colunas e os tipos de dados de cada tabela

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)


TRAIN
id                      int64
date           datetime64[ns]
store_nbr               int64
family                 object
sales                 float64
onpromotion             int64
dtype: object

STORES
store_nbr     int64
city         object
state        object
type         object
cluster       int64
dtype: object

TRANSACTIONS
date            datetime64[ns]
store_nbr                int64
transactions             int64
dtype: object

HOLIDAYS
date           datetime64[ns]
type                   object
locale                 object
locale_name            object
description            object
transferred              bool
dtype: object

OIL
date          datetime64[ns]
dcoilwtico           float64
dtype: object


In [7]:
# Visualizando uma amostra de cada tabela

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    display(df.head())


TRAIN


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0



STORES


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4



TRANSACTIONS


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922



HOLIDAYS


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False



OIL


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [8]:
# Verificando o período dos dados e a cardinalidade das principais dimensões

print(f"Período: {train['date'].min()} até {train['date'].max()}")
print(f"Quantidade de lojas: {train['store_nbr'].nunique()}")
print(f"Quantidade de famílias de produtos: {train['family'].nunique()}")

Período: 2013-01-01 00:00:00 até 2017-08-15 00:00:00
Quantidade de lojas: 54
Quantidade de famílias de produtos: 33


In [9]:
# Verificando duplicidades na granularidade principal da tabela de vendas

duplicate_count = train.duplicated(
    subset=["date", "store_nbr", "family"]
).sum()

print(f"Duplicidades em date + store_nbr + family: {duplicate_count}")

Duplicidades em date + store_nbr + family: 0


In [10]:
# Verificando duplicidades na granularidade da tabela de transações

duplicate_transactions = transactions.duplicated(
    subset=["date", "store_nbr"]
).sum()

print(f"Duplicidades em date + store_nbr: {duplicate_transactions}")

Duplicidades em date + store_nbr: 0


In [11]:
# Verificando quantas datas possuem mais de um registro na tabela de feriados

holiday_counts = (
    holidays.groupby("date")
    .size()
    .sort_values(ascending=False)
)

print(holiday_counts.head(10))

date
2014-06-25    4
2016-06-25    3
2012-06-25    3
2015-06-25    3
2017-06-25    3
2013-06-25    3
2015-12-22    2
2012-12-22    2
2012-12-24    2
2012-12-31    2
dtype: int64


In [12]:
# Inspecionando múltiplos registros de feriados para a mesma data

holidays[
    holidays["date"] == "2014-06-25"
]

,date,type,locale,locale_name,description,transferred
110,2014-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False
111,2014-06-25,Holiday,Local,Machala,Fundacion de Machala,False
112,2014-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False
113,2014-06-25,Event,National,Ecuador,Mundial de futbol Brasil: Ecuador-Francia,False


In [13]:
# Verificando cidades e estados disponíveis na tabela de lojas

print("Cidades:")
print(sorted(stores["city"].unique()))

print("\nEstados:")
print(sorted(stores["state"].unique()))

Cidades:
['Ambato', 'Babahoyo', 'Cayambe', 'Cuenca', 'Daule', 'El Carmen', 'Esmeraldas', 'Guaranda', 'Guayaquil', 'Ibarra', 'Latacunga', 'Libertad', 'Loja', 'Machala', 'Manta', 'Playas', 'Puyo', 'Quevedo', 'Quito', 'Riobamba', 'Salinas', 'Santo Domingo']

Estados:
['Azuay', 'Bolivar', 'Chimborazo', 'Cotopaxi', 'El Oro', 'Esmeraldas', 'Guayas', 'Imbabura', 'Loja', 'Los Rios', 'Manabi', 'Pastaza', 'Pichincha', 'Santa Elena', 'Santo Domingo de los Tsachilas', 'Tungurahua']


In [14]:
# Verificando correspondência entre localidades dos feriados e das lojas

local_holidays = set(
    holidays.loc[holidays["locale"] == "Local", "locale_name"].unique()
)

regional_holidays = set(
    holidays.loc[holidays["locale"] == "Regional", "locale_name"].unique()
)

store_cities = set(stores["city"].unique())
store_states = set(stores["state"].unique())

print("Locais sem correspondência:")
print(sorted(local_holidays - store_cities))

print("\nRegiões sem correspondência:")
print(sorted(regional_holidays - store_states))

Locais sem correspondência:
[]

Regiões sem correspondência:
[]


In [15]:
# Verificando valores ausentes em cada tabela

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.isna().sum())


TRAIN
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

STORES
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

TRANSACTIONS
date            0
store_nbr       0
transactions    0
dtype: int64

HOLIDAYS
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

OIL
date           0
dcoilwtico    43
dtype: int64


In [16]:
# Verificando continuidade temporal da tabela de petróleo
# Esse projeto original é do Equador, e o preço do petróleo pode funcionar como uma variável macroeconômica externa

oil_dates = pd.date_range(
    start=oil["date"].min(),
    end=oil["date"].max(),
    freq="D"
)

missing_oil_dates = oil_dates.difference(oil["date"])

print(f"Quantidade de datas ausentes em oil: {len(missing_oil_dates)}")
print(missing_oil_dates[:20])

Quantidade de datas ausentes em oil: 486
DatetimeIndex(['2013-01-05', '2013-01-06', '2013-01-12', '2013-01-13',
               '2013-01-19', '2013-01-20', '2013-01-26', '2013-01-27',
               '2013-02-02', '2013-02-03', '2013-02-09', '2013-02-10',
               '2013-02-16', '2013-02-17', '2013-02-23', '2013-02-24',
               '2013-03-02', '2013-03-03', '2013-03-09', '2013-03-10'],
              dtype='datetime64[ns]', freq=None)


In [17]:
# Verificando o período temporal disponível na tabela de transações

print(
    f"Período de vendas: "
    f"{train['date'].min().date()} até {train['date'].max().date()}"
)

print(
    f"Período de transações: "
    f"{transactions['date'].min().date()} até {transactions['date'].max().date()}"
)

Período de vendas: 2013-01-01 até 2017-08-15
Período de transações: 2013-01-01 até 2017-08-15


In [18]:
# Verificando a cobertura de transações para as combinações de data e loja presentes nas vendas

train_date_store = train[["date", "store_nbr"]].drop_duplicates()

transaction_coverage = train_date_store.merge(
    transactions[["date", "store_nbr"]],
    on=["date", "store_nbr"],
    how="left",
    indicator=True
)

print(transaction_coverage["_merge"].value_counts())

_merge
both          83488
left_only      7448
right_only        0
Name: count, dtype: int64


In [19]:
# Inspecionando combinações de data e loja sem registro de transações

missing_transactions = transaction_coverage.loc[
    transaction_coverage["_merge"] == "left_only",
    ["date", "store_nbr"]
]

display(missing_transactions.head(20))

,date,store_nbr
0,2013-01-01,1
1,2013-01-01,10
2,2013-01-01,11
3,2013-01-01,12
4,2013-01-01,13
5,2013-01-01,14
6,2013-01-01,15
7,2013-01-01,16
8,2013-01-01,17
9,2013-01-01,18


In [20]:
# Verificando as datas com maior quantidade de lojas sem registro de transações

missing_transactions_by_date = (
    missing_transactions
    .groupby("date")
    .size()
    .sort_values(ascending=False)
)

display(missing_transactions_by_date.head(20))

date
2016-01-01    54
2016-01-03    54
2017-01-01    53
2015-01-01    53
2013-01-01    53
2014-01-01    52
2016-01-04    40
2016-01-02    18
2013-06-19    11
2013-01-31     8
2013-03-04     8
2013-03-05     8
2013-02-01     8
2013-03-03     8
2013-02-04     8
2013-02-05     8
2013-02-06     8
2013-02-07     8
2013-02-08     8
2013-02-09     8
dtype: int64

In [21]:
# Verificando vendas nas combinações de data e loja sem registro de transações

missing_transactions_sales = (
    train
    .merge(
        missing_transactions,
        on=["date", "store_nbr"],
        how="inner"
    )
    .groupby(["date", "store_nbr"], as_index=False)["sales"]
    .sum()
)

display(
    missing_transactions_sales
    .sort_values("sales", ascending=False)
    .head(20)
)

,date,store_nbr,sales
6692,2016-01-03,45,74597.100000
6691,2016-01-03,44,74026.947000
6694,2016-01-03,47,67248.303000
6693,2016-01-03,46,58926.382000
6650,2016-01-03,3,56595.148000
6695,2016-01-03,48,54569.516998
6696,2016-01-03,49,53416.107020
6731,2016-01-04,44,52673.883070
6732,2016-01-04,45,50025.920000
6703,2016-01-04,3,47439.963000


In [22]:
# Verificando a ocorrência de vendas nas combinações sem registro de transações

print(
    "Combinações com vendas > 0:",
    (missing_transactions_sales["sales"] > 0).sum()
)

print(
    "Combinações com vendas = 0:",
    (missing_transactions_sales["sales"] == 0).sum()
)

Combinações com vendas > 0: 118
Combinações com vendas = 0: 7330


In [23]:
# Verificando a quantidade de registros de vendas por dia

rows_per_day = (
    train.groupby("date")
    .size()
)

print(rows_per_day.describe())

count    1684.0
mean     1782.0
std         0.0
min      1782.0
25%      1782.0
50%      1782.0
75%      1782.0
max      1782.0
dtype: float64


In [24]:
# Verificando datas ausentes no histórico de vendas

expected_dates = pd.date_range(
    start=train["date"].min(),
    end=train["date"].max(),
    freq="D"
)

missing_sales_dates = expected_dates.difference(train["date"].unique())

print(f"Quantidade de datas ausentes: {len(missing_sales_dates)}")
print(missing_sales_dates)

Quantidade de datas ausentes: 4
DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'], dtype='datetime64[ns]', freq=None)


In [25]:
# Verificando correspondência das lojas entre as tabelas de vendas e cadastro

train_stores = set(train["store_nbr"].unique())
registered_stores = set(stores["store_nbr"].unique())

print("Lojas em train sem cadastro em stores:")
print(sorted(train_stores - registered_stores))

print("\nLojas em stores sem vendas em train:")
print(sorted(registered_stores - train_stores))

Lojas em train sem cadastro em stores:
[]

Lojas em stores sem vendas em train:
[]


In [26]:
# Carregando e comparando a estrutura dos conjuntos de treino e teste

test = pd.read_csv(
    DATA_PATH / "test.csv",
    parse_dates=["date"]
)

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")

print("\nColunas de train:")
print(train.columns.tolist())

print("\nColunas de test:")
print(test.columns.tolist())

Train: (3000888, 6)
Test: (28512, 5)

Colunas de train:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']

Colunas de test:
['id', 'date', 'store_nbr', 'family', 'onpromotion']


In [27]:
# Verificando o período temporal do conjunto de teste

print(f"Período do test: {test['date'].min()} até {test['date'].max()}")
print(f"Quantidade de dias: {test['date'].nunique()}")

Período do test: 2017-08-16 00:00:00 até 2017-08-31 00:00:00
Quantidade de dias: 16


## Conclusões do inventário

- O histórico de vendas contém 3.000.888 registros entre 01/01/2013 e 15/08/2017.
- A base possui 54 lojas e 33 famílias de produtos.
- A granularidade da tabela de vendas é `date + store_nbr + family`, sem duplicidades.
- Todos os dias presentes no histórico possuem as 1.782 combinações esperadas de loja e família.
- As únicas datas ausentes no histórico de vendas são os dias 25 de dezembro de 2013 a 2016.
- Todas as lojas presentes em `train` possuem correspondência na tabela `stores`.
- A tabela `transactions` possui granularidade `date + store_nbr`, sem duplicidades, mas não cobre todas as combinações presentes em vendas.
- Existem 7.448 combinações de data e loja sem registro de transações; 118 delas possuem vendas positivas, portanto a ausência de `transactions` não pode ser interpretada automaticamente como zero.
- A tabela de feriados possui eventos nacionais, regionais e locais. Eventos regionais e locais possuem correspondência com os estados e cidades cadastrados em `stores`.
- A tabela `oil` possui 43 valores ausentes e lacunas no calendário, sendo considerada uma variável externa opcional neste projeto.
- O conjunto de teste cobre 16 dias, de 16/08/2017 a 31/08/2017, e não contém a variável-alvo `sales`.